# Training external machinery against TorchLens surfaces

TorchLens deliberately ships **no trainers**: no probe fitters, no SAE
training loops, no steering-vector optimizers. What it ships instead are the
surfaces such machinery trains *against*:

1. **Extracted activations** feed any external probe trainer (Part 1).
2. **`spec.bind(model)`** gives a capture-free intervened runner -- real
   outputs, real gradients -- that an external optimizer can drive at roughly
   native forward cost (Part 2: steering-vector optimization).
3. **`bridge.sae.splice`** measures a *trained* SAE causally (Part 3,
   pointer).

Everything below is offline, seeded, CPU-only, and runs in well under a
minute. Every spelling used here is DOCUMENTED-UNSTABLE pending the naming
session; the workflows are the stable part.


## Setup: a small model with a planted signal

Two-class data in 8 dimensions whose class signal lies along one planted
direction, and a small MLP trained inline to ~99% accuracy. No downloads.


In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

import torchlens as tl

torch.manual_seed(7)

N, D, H = 512, 8, 16
signal = torch.randn(D)
signal /= signal.norm()
y = torch.randint(0, 2, (N,))
X = torch.randn(N, D) * 0.7 + torch.outer(y.float() * 2 - 1, signal) * 1.5


class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(D, H)
        self.act = nn.ReLU()
        self.fc2 = nn.Linear(H, 2)

    def forward(self, x):
        return self.fc2(self.act(self.fc1(x)))


model = MLP()
opt = torch.optim.Adam(model.parameters(), lr=1e-2)
for _ in range(150):
    opt.zero_grad()
    F.cross_entropy(model(X), y).backward()
    opt.step()
model.eval()

model_acc = (model(X).argmax(-1) == y).float().mean()
print(f"model accuracy: {model_acc:.3f}")

model accuracy: 0.990


## Part 1 -- train a linear probe on extracted activations

One capture retains the hidden ReLU activations; the probe is a plain
`nn.Linear` trained by an external `torch.optim.Adam`. TorchLens's role ends
at handing over honest activations -- the trainer is yours. (For dataset-scale
extraction with manifests, resume, and disk shards, use
`tl.extract_dataset`; one in-memory batched capture is enough here.)


In [2]:
log = tl.trace(model, X, save=tl.func("relu"))
acts = log["relu_1_2"].out.detach()  # [N, H], detached payload
print("extracted activations:", tuple(acts.shape))

probe = nn.Linear(H, 2)
popt = torch.optim.Adam(probe.parameters(), lr=1e-2)
for _ in range(200):
    popt.zero_grad()
    F.cross_entropy(probe(acts), y).backward()
    popt.step()

probe_acc = (probe(acts).argmax(-1) == y).float().mean()
print(f"probe accuracy on the hidden site: {probe_acc:.3f}")

# the probe's class-1 decoding direction in activation space
probe_direction = (probe.weight[1] - probe.weight[0]).detach()

extracted activations: (512, 16)


probe accuracy on the hidden site: 0.992


## Part 2 -- optimize a steering vector through `spec.bind`

The steering vector `v` is an ordinary leaf tensor owned by an external
optimizer. `tl.when(site, tl.steer(v, ...))` is an `InterventionSpec`, and
`spec.bind(model)` returns a runner that executes the model with the edit
applied at the site -- **capture-free**, real outputs, and gradients flow
back to `v` (the helper holds `v` by reference, so each optimizer step is
seen by the next call).

The training goal: flip class-0 inputs to class-1 predictions with a small
L2 penalty keeping `v` minimal.


In [3]:
v = torch.zeros(H, requires_grad=True)
spec = tl.when(tl.func("relu"), tl.steer(v, magnitude=1.0, feature_axis=-1))
runner = spec.bind(model)

X0 = X[y == 0]
target = torch.ones(len(X0), dtype=torch.long)
sopt = torch.optim.Adam([v], lr=5e-2)

for _ in range(120):
    sopt.zero_grad()
    out = runner(X0)  # intervened forward, ~native cost
    loss = F.cross_entropy(out, target) + 0.01 * v.norm() ** 2
    loss.backward()  # gradients flow through the edit into v
    sopt.step()

baseline_rate = (model(X0).argmax(-1) == 1).float().mean()
steered_rate = (runner(X0).argmax(-1) == 1).float().mean()
print(f"class-1 rate on class-0 inputs: baseline {baseline_rate:.3f} -> steered {steered_rate:.3f}")

class-1 rate on class-0 inputs: baseline 0.012 -> steered 0.984


### The payoff: the optimizer rediscovers the probe's direction

Two *independent* pieces of external machinery -- a supervised probe reading
the site, and a causal steering optimizer writing to it -- agree about where
the class lives in activation space:


In [4]:
cos = F.cosine_similarity(v.detach(), probe_direction, dim=0)
print(f"cosine(steering vector, probe direction) = {cos:.3f}")

cosine(steering vector, probe direction) = 0.920


### Verify the intervention honestly with one capture

`spec.bind` is deliberately capture-free for speed. To *prove* what fired
where, run the same spec once under `tl.trace`: the intervention audit
records the fired edit at the site, and the saved activations show the
steered values.

One honesty note: the audit's `edit` string reprs the helper's arguments as
constructed (here: the zero initialization of `v`); execution always reads
the live tensor, as the flipped predictions above demonstrate.


In [5]:
vlog = tl.trace(model, X0[:4], intervene=spec, save=tl.func("relu"))
print("audit entries:", len(vlog.intervention_audit))

steered_acts = vlog["relu_1_2"].out
clean_acts = tl.trace(model, X0[:4], save=tl.func("relu"))["relu_1_2"].out
print(
    "mean site delta vs clean capture:",
    f"{(steered_acts - clean_acts).mean():.4f}",
    "(expected: mean of v =",
    f"{v.detach().mean():.4f})",
)

audit entries: 1
mean site delta vs clean capture: -0.0732 (expected: mean of v = -0.0732)


## Part 3 -- trained SAEs: measure them causally with `bridge.sae.splice`

The same boundary applies to sparse autoencoders: train one with any
external library (SAELens or your own loop -- TorchLens ships neither), then
hand the trained artifact to TorchLens to *measure*:

```python
result = tl.bridge.sae.splice(model, x, site=..., sae=trained_sae)
result_edited = tl.bridge.sae.splice(
    model, x, site=..., sae=trained_sae,
    latents_edit=lambda z: z.clamp(min=0) * mask,  # per-feature causal knob
)
```

`splice` swaps the SAE's reconstruction in at the site (fork +
`tl.splice_module` + replay) and reports reconstruction fidelity plus the
output-level causal effect. The SAE only needs `encode`/`decode` -- no SAE
package is required. The full walkthrough (including an untrained-SAE
baseline) is `notebooks/sae_splice_tutorial.ipynb`.

## Where the boundary sits

| You bring | TorchLens provides |
| --- | --- |
| Probe architecture + optimizer | honest extracted activations (`tl.trace`, `tl.extract_dataset`) |
| Steering objective + optimizer | a gradient-transparent intervened runner (`spec.bind`) and the audited capture (`intervene=`) |
| A trained SAE | causal measurement at a site (`bridge.sae.splice`) |

TorchLens stays a measurement instrument; the trainable machinery -- and
credit for it -- belongs to the ecosystem it feeds
(see `docs/acknowledgments.md`).


Related: `examples/recipes/trainable_intervention_recipe.ipynb` learns the same kind of additive vector through the *replay* engine (fork + hook + backprop through the replayed output) instead of the bound runner -- the right tool when you want to reuse one captured graph rather than re-run the model.